<a href="https://colab.research.google.com/github/fauziardiantama/my-falcon-tuning/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Falcon 90M Training & Auto-Transfer

Notebook ini dikonfigurasi untuk menjalankan training pada Falcon 90M Instruct (Full FT atau LoRA) dan mengirimkan hasilnya ke komputer lokal menggunakan `croc`.

In [1]:
# 1. Clone Repositori dan Pindah ke Direktori
import os
REPO_URL = 'https://github.com/fauziardiantama/my-falcon-tuning.git'
REPO_NAME = 'my-falcon-tuning'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# --- DIAGNOSTICS ---
print("--- SYSTEM DIAGNOSTICS ---")
!nvcc --version
!python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA available: {torch.cuda.is_available()}'); print(f'CUDA version: {torch.version.cuda}')"

# 2. Instalasi Requirements
TRAINING_METHOD = 'full' # 'full' atau 'lora'
REQ_FILE = 'requirements_lora.txt' if TRAINING_METHOD == 'lora' else 'requirements.txt'

print(f"--- INSTALLING DEPENDENCIES FOR {TRAINING_METHOD.upper()} ---")
# Instalasi standar untuk requirements umum
!pip install -q -r {REQ_FILE}

# Instalasi khusus library akselerasi dengan penanganan error yang lebih baik
if TRAINING_METHOD == 'full':
    print("--- INSTALLING MAMBA ACCELERATION KERNELS ---")
    # Menggunakan MAX_JOBS untuk mencegah kehabisan RAM saat kompilasi
    # --no-cache-dir memastikan kita tidak menggunakan build yang rusak sebelumnya
    !MAX_JOBS=4 pip install --no-cache-dir causal-conv1d>=1.4.0 mamba-ssm

# 3. Instalasi Croc (File Transfer)
print("--- INSTALLING CROC ---")
!curl https://getcroc.schollz.com | bash

Cloning into 'my-falcon-tuning'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 57 (delta 30), reused 38 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 3.69 MiB | 18.08 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/my-falcon-tuning
--- INSTALLING DEPENDENCIES FOR FULL ---
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d 

In [2]:
# 4. Jalankan Script Training
TRAINING_SCRIPT = 'train_falcon_lora.py' if TRAINING_METHOD == 'lora' else 'train_falcon.py'
print(f"--- STARTING {TRAINING_METHOD.upper()} TRAINING ---")
!python {TRAINING_SCRIPT}

--- STARTING FULL TRAINING ---
Loading tokenizer: tiiuae/Falcon-H1-Tiny-90M-Instruct
config.json: 1.62kB [00:00, 2.86MB/s]
tokenizer_config.json: 99.7kB [00:00, 129MB/s]
tokenizer.json: 2.35MB [00:00, 94.1MB/s]
special_tokens_map.json: 7.69kB [00:00, 19.5MB/s]
chat_template.jinja: 2.71kB [00:00, 8.12MB/s]
Loading model: tiiuae/Falcon-H1-Tiny-90M-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 182M/182M [00:02<00:00, 87.0MB/s]
The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 386/386 [00:00<00:00, 1054.70it/s, Materializing param=model.layers.23.self_attn.v_proj.weight]
generation_config.json: 100% 152/152 [00:00<00:00, 792kB/s]
Loading dataset from: dataset_rpg.jsonl
Generating train split: 24260 exampl

In [3]:
# 5. Kirim Hasil Menggunakan Croc
OUTPUT_DIR = './falcon-90m-lora-adapter' if TRAINING_METHOD == 'lora' else './falcon-90m-fine-tuned'
print(f"--- SENDING {TRAINING_METHOD.upper()} RESULT VIA CROC ---")
print("Salin kode yang muncul di bawah ini untuk menerima file di komputer lokal:")
!croc send {OUTPUT_DIR}

--- SENDING FULL RESULT VIA CROC ---
Salin kode yang muncul di bawah ini untuk menerima file di komputer lokal:
Sending 0 files and 1 folders (0 B)
Code is: 8330-camera-exile-ibiza

On the other computer run:
(For Windows)
    croc 8330-camera-exile-ibiza
(For Linux/macOS)
    CROC_SECRET="8330-camera-exile-ibiza" croc 
